# Medical Insurance Charges — Linear Regression Practice

This is a hands-on learning notebook using the Medical Insurance dataset. It records our current progress in data inspection, basic EDA, a mean baseline, and a first Linear Regression model.

**Status:** In progress. The broader Linear Regression study will continue later.

**Dataset:** [insurance.csv](insurance.csv) (included in this folder).

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("insurance.csv")
df.head()

## 2. Inspect data quality and structure

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)]

In [ ]:
df = df.drop_duplicates()
print(df.shape)
print(df.duplicated().sum())

## 3. Exploratory data analysis

### Average charges by smoking status

Business question: Are average insurance charges different for smokers and non-smokers?

In [ ]:
df.groupby("smoker")["charges"].mean()

### Age vs. insurance charges

Business question: Do insurance charges tend to increase as age increases?

In [ ]:
sns.scatterplot(data=df, x="age", y="charges")
plt.title("Age and Insurance Charges")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.show()

### BMI vs. insurance charges

Business question: Do customers with higher BMI tend to have higher insurance charges? Color by smoking status to compare the groups.

In [ ]:
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker")
plt.title("BMI and Insurance Charges by Smoking Status")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.show()

## 4. Prepare features and target

For this first model, use only the numerical features `age`, `bmi`, and `children`. Categorical features such as `smoker` and `region` have not been encoded yet; we'll cover that later.

In [ ]:
X = df[["age", "bmi", "children"]]
y = df["charges"]

print(X.shape)
print(y.shape)

## 5. Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 6. Baseline model

The mean baseline predicts the average charge from the training target for every test row. We'll compare our model against this simple reference.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("Baseline MAE:", baseline_mae)

## 7. Linear Regression model

Train a Linear Regression model using the same train-test split and evaluate it with Mean Absolute Error (MAE).

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print("Linear Regression MAE:", mae)

## 8. Compare actual and predicted charges

In [ ]:
comparison = pd.DataFrame({
    "Actual Charges": y_test.values,
    "Predicted Charges": y_pred
})
comparison["Error"] = comparison["Actual Charges"] - comparison["Predicted Charges"]
comparison.head(10)

## 9. Investigate large prediction errors

Inspect a couple of test customers with large errors and check their original feature values.

In [ ]:
rows_to_check = y_test.iloc[[3, 6]].index

df.loc[
    rows_to_check,
    ["age", "bmi", "children", "smoker", "charges"]
]

## 10. Actual vs. predicted plot

The red dashed line represents perfect predictions. Points below the line are underpredictions; points above it are overpredictions.

In [ ]:
sns.scatterplot(x=y_test, y=y_pred)
plt.title("Actual vs. Predicted Insurance Charges")
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--"
)
plt.show()

## Current learning notes

- The dataset initially contains 1,338 rows and 7 columns.
- We found one exact duplicate row and removed the duplicate.
- Average charges in this dataset were higher for smokers than non-smokers; this is an observed association, not by itself proof of causation.
- Current features: `age`, `bmi`, and `children` only.
- On the current 80/20 split (`random_state=42`), the baseline MAE was about **9,861.80** and Linear Regression MAE was about **9,620.93**.
- The improvement over the baseline is modest. The plot and error inspection show the current model underpredicts some high-charge customers.
- This is an introductory practice exercise, not a completed end-to-end regression study. R², categorical encoding, pipelines, and further model improvements are still to be covered.